# GRPO Training — Poker Decision Advisor (Vast.ai)

Fine-tunes Qwen3-8B with GRPO on the PokerBench dataset.

**SFT adapter must be at:** `/workspace/sft-adapter/`

In [ ]:
import sys
!{sys.executable} -m pip install datasets bitsandbytes peft accelerate
!{sys.executable} -m pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!{sys.executable} -m pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

## 2. GPU check

In [1]:
import torch
assert torch.cuda.is_available(), "No GPU found"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [2]:
import sys
print(sys.executable)

/venv/main/bin/python


## 3. Set paths

In [3]:
import os

SFT_ADAPTER_DIR = "/workspace/sft-adapter"
CHECKPOINT_DIR  = "/workspace/grpo-checkpoints"
OUTPUT_DIR      = "/workspace/grpo-adapter"
DATA_CACHE_DIR  = "/workspace/data"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

assert os.path.exists(SFT_ADAPTER_DIR), f"SFT adapter not found at {SFT_ADAPTER_DIR}"
print(f"SFT adapter : {SFT_ADAPTER_DIR}")
print(f"Contents    : {os.listdir(SFT_ADAPTER_DIR)}")
print(f"Checkpoints : {CHECKPOINT_DIR}")
print(f"Output      : {OUTPUT_DIR}")

SFT adapter : /workspace/sft-adapter
Contents    : ['chat_template.jinja', 'README.md', 'tokenizer_config.json', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors']
Checkpoints : /workspace/grpo-checkpoints
Output      : /workspace/grpo-adapter


## 4. Load dataset

In [4]:
from datasets import load_dataset

dataset = load_dataset("RZ412/PokerBench", cache_dir=DATA_CACHE_DIR)
train_ds = dataset["train"]
test_ds  = dataset["test"]
print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")

Train: 563,200  Test: 11,000


## 5. Preprocessor

In [5]:
SYSTEM_PROMPT = (
    "You are a poker decision engine. Given a game scenario, output only the "
    "optimal action (check, fold, call, bet X, or raise X). Do not explain."
)


def format_grpo(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["instruction"]},
    ]


def apply_chat_template(messages, tokenizer, add_generation_prompt=False):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )


print("Preprocessor ready")

Preprocessor ready


## 6. Reward function (tiered scoring)

In [6]:
import re

VALID_ACTIONS = ("check", "fold", "call", "bet", "raise")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_PASSIVE = {"check", "call"}
_AGGRESSIVE = {"bet", "raise"}


def _strip_thinking(text):
    return _THINK_RE.sub("", text).strip()


def parse_action_type(text):
    text = _strip_thinking(text).lower()
    for action in VALID_ACTIONS:
        if text.startswith(action):
            return action
    if "all-in" in text or "allin" in text or "all in" in text:
        return "raise"
    return None


def parse_bet_amount(text):
    text = _strip_thinking(text).lower()
    action = parse_action_type(text)
    if action in ("check", "fold", "call"):
        return None
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else None


def poker_reward(predicted, correct):
    pred_action = parse_action_type(predicted)
    true_action = parse_action_type(correct)
    if pred_action != true_action:
        both = {pred_action, true_action}
        if both <= _AGGRESSIVE:
            return -0.3
        if both <= _PASSIVE:
            return -0.3
        return -1.0
    true_amount = parse_bet_amount(correct)
    if true_amount is None:
        return 1.0
    if true_amount == 0:
        return 1.0
    pred_amount = parse_bet_amount(predicted)
    if pred_amount is None:
        return 0.1
    ratio = pred_amount / true_amount
    if 0.9 <= ratio <= 1.1:
        return 1.0
    elif 0.8 <= ratio <= 1.2:
        return 0.7
    elif 0.5 <= ratio <= 1.5:
        return 0.4
    else:
        return 0.1


assert poker_reward("bet 18", "bet 18") == 1.0
assert poker_reward("bet 20", "bet 18") == 0.7
assert poker_reward("fold", "raise 10") == -1.0
assert poker_reward("bet 10", "raise 10") == -0.3
assert poker_reward("fold", "fold") == 1.0
print("Reward function OK")

Reward function OK


## 7. Load model

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
print(f"Model loaded — VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.566 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.5 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded — VRAM: 7.9 GB


## 8. Preprocess dataset — filtered to hard examples

SFT scores 89% overall but only 70% on bet and 85% on raise. Filter to these to maximise GRPO learning signal.

In [8]:
def preprocess_for_grpo(row):
    return {
        "prompt": apply_chat_template(
            format_grpo(row), tokenizer, add_generation_prompt=True
        ),
        "answer": row["output"],
    }


hard_train_ds = train_ds.filter(
    lambda row: row["output"].strip().lower().split()[0] in ("bet", "raise")
)
print(f"Full train set        : {len(train_ds):,}")
print(f"Hard subset (bet/raise): {len(hard_train_ds):,} ({len(hard_train_ds)/len(train_ds)*100:.1f}%)")

grpo_dataset = hard_train_ds.map(
    preprocess_for_grpo,
    remove_columns=hard_train_ds.column_names,
)
print(f"Preprocessed {len(grpo_dataset):,} rows")

Full train set        : 563,200
Hard subset (bet/raise): 133,000 (23.6%)
Preprocessed 133,000 rows


## 9. GRPO reward wrapper

In [9]:
def grpo_reward_fn(completions, answer=None, **kwargs):
    return [poker_reward(c, a) for c, a in zip(completions, answer)]


test_result = grpo_reward_fn(["bet 18", "fold"], answer=["bet 18", "raise 10"])
assert test_result == [1.0, -1.0]
print("GRPO reward wrapper OK")

GRPO reward wrapper OK


## 10. Completion logger

In [18]:
from transformers import TrainerCallback

class MetricsLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        reward = logs.get("reward")
        if reward is not None:
            loss = logs.get("loss", 0)
            reward_std = logs.get("reward_std", 0)
            kl = logs.get("kl", 0)
            comp_len = logs.get("completion_length", 0)
            print(f"Step {step}: loss={loss:.4f} | reward={reward:.3f} ± {reward_std:.3f} | kl={kl:.3f} | comp_len={comp_len:.1f}")


class CompletionLogger(TrainerCallback):
    def on_step_end(self, cb_args, state, control, **kwargs):
        if state.global_step % 100 == 0:
            model_ref = kwargs.get("model")
            if model_ref is None:
                return
            sample = grpo_dataset[0]
            inputs = tokenizer(sample["prompt"], return_tensors="pt").to("cuda")
            with torch.no_grad():
                completions = []
                for _ in range(3):
                    out = model_ref.generate(
                        **inputs,
                        max_new_tokens=16,
                        do_sample=True,
                        temperature=0.7,
                        pad_token_id=tokenizer.eos_token_id,
                        use_cache=False,  # <-- add this
                    )
                    gen = tokenizer.decode(
                        out[0][inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    ).strip()
                    reward = poker_reward(gen, sample["answer"])
                    completions.append(f"'{gen}' -> {reward:.1f}")
            print(f"Step {state.global_step} samples: {' | '.join(completions)} (answer: '{sample['answer']}')")

print("Callbacks ready")

Callbacks ready


## 11. Configure and run GRPO training

**Adjust MAX_STEPS and BATCH_SIZE based on your GPU:**
- T4 (16GB): BATCH_SIZE=1, NUM_GENERATIONS=4
- A100 (40GB): BATCH_SIZE=2, NUM_GENERATIONS=6

In [14]:
import sys, subprocess, types

# Step 1: Reinstall TRL clean
subprocess.run([sys.executable, "-m", "pip", "install", "trl==0.24.0", "--force-reinstall", "--no-deps", "--no-cache-dir", "-q"], check=True)
print("Reinstalled TRL 0.24.0 clean")

# Step 2: Create dummy modules BEFORE importing TRL
for mod_name in ["vllm", "vllm.sampling_params", "mergekit", "mergekit.config",
                  "mergekit.merge", "llm_blender", "weave", "weave.trace",
                  "weave.trace.context"]:
    sys.modules[mod_name] = types.ModuleType(mod_name)

sys.modules["vllm"].LLM = None
sys.modules["vllm"].SamplingParams = None
sys.modules["mergekit.config"].MergeConfiguration = None
sys.modules["mergekit.merge"].MergeOptions = None
sys.modules["mergekit.merge"].run_merge = None
sys.modules["weave"].EvaluationLogger = None
sys.modules["weave.trace.context"].weave_client_context = None

from trl import GRPOTrainer, GRPOConfig
print(f"TRL {__import__('trl').__version__} imported successfully")

Reinstalled TRL 0.24.0 clean
TRL 0.24.0 imported successfully


In [ ]:
from trl import GRPOTrainer, GRPOConfig
import time

# ── Auto-detect GPU and set batch size ──
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb > 30:  # A100 or similar
    BATCH_SIZE = 2
    NUM_GENERATIONS = 6
    GRAD_ACCUM = 4
    print(f"Large GPU detected ({vram_gb:.0f}GB) — using batch_size=2, num_generations=6")
else:  # T4 or similar
    BATCH_SIZE = 1
    NUM_GENERATIONS = 4
    GRAD_ACCUM = 8
    print(f"Small GPU detected ({vram_gb:.0f}GB) — using batch_size=1, num_generations=4")

MAX_STEPS     = 1000
LEARNING_RATE = 4e-5
BETA          = 0.05
MAX_GRAD_NORM = 0.3

use_bf16 = torch.cuda.is_bf16_supported()
print(f"Precision: {'bf16' if use_bf16 else 'fp16'}")

config = GRPOConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    beta=BETA,
    num_generations=NUM_GENERATIONS,
    max_completion_length=48,
    max_prompt_length=512,
    max_grad_norm=MAX_GRAD_NORM,
    temperature=0.7,
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=False,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
)

model.warnings_issued = {"estimate_tokens": True}

# Prevent OOM from deepcopy of ref model (PEFT disables adapter instead)
import trl.trainer.grpo_trainer as _grpo
_orig_init = _grpo.GRPOTrainer.__init__
def _patched_init(self, *args, **kwargs):
    _orig_init(self, *args, **kwargs)
_grpo.create_reference_model = lambda model, *a, **kw: None


trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_fn,
    args=config,
    train_dataset=grpo_dataset,
)
trainer.add_callback(MetricsLogger())
trainer.add_callback(CompletionLogger())
trainer.generation_config.use_cache = False

effective_batch = BATCH_SIZE * GRAD_ACCUM * NUM_GENERATIONS
print(f"Training set    : {len(grpo_dataset):,} rows (bet/raise only)")
print(f"Effective batch : {effective_batch} completions per update")
print(f"Steps           : {MAX_STEPS}")

# Resume from checkpoint if one exists
checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")] \
    if os.path.exists(CHECKPOINT_DIR) else []
resume_from = CHECKPOINT_DIR if checkpoints else None
if resume_from:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    print(f"Resuming from: {latest}")
else:
    print("Starting from scratch")

t0 = time.time()
trainer_stats = trainer.train(resume_from_checkpoint=resume_from)
elapsed = time.time() - t0
print(f"\nTraining complete — {elapsed / 60:.1f} min")
print(f"Loss: {trainer_stats.metrics['train_loss']:.4f}")

Small GPU detected (16GB) — using batch_size=1, num_generations=4
Precision: fp16
Training set    : 133,000 rows (bet/raise only)
Effective batch : 32 completions per update
Steps           : 1000
Starting from scratch


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 133,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)
Both `max_new_tokens` (=48) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/venv/main/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/venv/main/lib/

Step,Training Loss
1,0.156202
2,0.255728
3,0.301071
4,0.299103
5,0.351837
6,0.280480
7,0.261802
8,0.321733
9,0.377920
10,0.236157


Both `max_new_tokens` (=48) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: loss=0.1562 | reward=1.000 ± 0.000 | kl=3.191 | comp_len=0.0


...

Step 667: loss=0.3690 | reward=0.550 ± 0.000 | kl=7.412 | comp_len=0.0


Both `max_new_tokens` (=48) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 668: loss=0.3650 | reward=1.000 ± 0.000 | kl=7.267 | comp_len=0.0


Both `max_new_tokens` (=48) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 669: loss=0.2419 | reward=1.000 ± 0.000 | kl=4.838 | comp_len=0.0


## 12. Save adapter

In [16]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

Saved to /workspace/grpo-adapter
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


## 13. Spot check (20 examples)

In [17]:
FastLanguageModel.for_inference(model)

spot_test = test_ds.select(range(20))
correct = 0

for row in spot_test:
    prompt = apply_chat_template(
        format_grpo(row), tokenizer, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                 do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False,
                )
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    generated_clean = _strip_thinking(generated)
    reward = poker_reward(generated_clean, row["output"])
    if reward == 1.0:
        correct += 1
    print(f"Expected: {row['output']:<12} Predicted: {generated_clean:<12} Reward: {reward}")

print(f"\nSpot-check: {correct}/20 ({correct * 5}%)")

Both `max_new_tokens` (=32) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RuntimeError: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 297, 128]

## 14. Download adapter

Push to HuggingFace Hub so you can access it from anywhere:

In [ ]:
# Uncomment and set your token:

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
# model.push_to_hub("dominicvdb/poker-grpo-adapter")
# tokenizer.push_to_hub("dominicvdb/poker-grpo-adapter")
# print("Pushed to HuggingFace Hub")

In [ ]:
# import sys
# import subprocess

# # Force install TRL 0.14.0 which has no vllm/mergekit/weave/llm_blender deps
# subprocess.run([sys.executable, "-m", "pip", "install", "trl==0.14.0", "--force-reinstall", "--no-deps"], check=True)
# print("Installed TRL 0.14.0")

# # Clear all cached TRL imports  
# for key in list(sys.modules.keys()):
#     if "trl" in key:
#         del sys.modules[key]

# # Restart needed — cached modules won't fully clear
# import os
# os._exit(0)